# ABLE Distillation: Fine-tune Qwen/Qwen3.5-9B

Generated: 2026-04-07 07:20 UTC

**Model**: Qwen/Qwen3.5-9B (edge)
**Corpus**: `~/.able/distillation/corpus/default/latest/train.jsonl`
**Runtime**: t4_colab
**Epochs**: 3

This notebook uses Unsloth for 2x faster training with 70% less VRAM.
Free Colab T4 runtime: 12-24 hours available.

In [ ]:
%%capture
# Install Unsloth (takes ~2 minutes on Colab)
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-9B",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,  # Auto-detect (float16 on T4, bfloat16 on A100/H100)
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    use_gradient_checkpointing="unsloth",
)

print(f"Model loaded: {model.get_nb_trainable_parameters()} trainable params")

In [ ]:
from datasets import load_dataset
import json

# Load ABLE distillation corpus (ChatML format)
# Upload train.jsonl to Colab or mount Google Drive
CORPUS_PATH = "train.jsonl  # Upload your corpus JSONL here, or mount Google Drive"

def format_chatml(example):
    """Format a single training example as ChatML."""
    messages = example.get('messages', [])
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": formatted}

dataset = load_dataset("json", data_files=CORPUS_PATH, split="train")
dataset = dataset.map(format_chatml)
print(f"Loaded {len(dataset)} training examples")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        learning_rate=0.0002,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        save_strategy="steps",
        save_steps=100,
        output_dir="outputs/able-nano-9b",
        optim="adamw_8bit",
        warmup_steps=5,
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)

## Export to GGUF for Ollama

This exports the fine-tuned model to GGUF format using Unsloth's
Dynamic 2.0 quantization for optimal size/quality balance.

In [ ]:
# Export to GGUF (Unsloth Dynamic 2.0 quantization)
QUANT_METHODS = ["q4_k_m", "iq2_m", "q8_0"]

for quant in QUANT_METHODS:
    print(f"Exporting {quant}...")
    model.save_pretrained_gguf(
        f"outputs/able-nano-9b-gguf",
        tokenizer,
        quantization_method=quant,
    )
    print(f"  Done: outputs/able-nano-9b-gguf")

# Optional: Push to HuggingFace Hub
# model.push_to_hub_gguf(
#     "able-distilled/able-nano-9b",
#     tokenizer,
#     quantization_method=QUANT_METHODS[0],
#     token="hf_...",  # Your HF token
# )

## Deploy to Ollama

After downloading the GGUF file to your local machine:

In [ ]:
# Generate Ollama Modelfile
MODELFILE = '''
FROM ./outputs/able-nano-9b-gguf/unsloth.Q4_K_M.gguf
TEMPLATE """{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{- end }}<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
{{ .Response }}<|im_end|>"""
PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
SYSTEM You are ABLE, an autonomous AI agent.
'''

with open("Modelfile", "w") as f:
    f.write(MODELFILE)

print("Modelfile generated. Run locally:")
print(f"  ollama create able-nano-9b -f Modelfile")
print(f"  ollama run able-nano-9b")

## Training Stats for Federation

After training, the federation sync will pick up these metrics
and share quality improvements across the network.

In [ ]:
import json
from datetime import datetime, timezone

stats = {
    "model": "able-nano-9b",
    "base": "Qwen/Qwen3.5-9B",
    "corpus_path": CORPUS_PATH,
    "corpus_size": len(dataset),
    "epochs": 3,
    "runtime": "t4_colab",
    "quants": QUANT_METHODS,
    "loss": trainer_stats.training_loss,
    "completed_at": datetime.now(timezone.utc).isoformat(),
}

with open("outputs/able-nano-9b_training_stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print(json.dumps(stats, indent=2))